<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microscopy-Core-ISMMS/ImageAnalysisCourse/blob/2026-workshop/notebooks/01_cellpose_segmentation.ipynb)

*Click the badge to open this notebook in Google Colab. For best performance, switch to a GPU runtime: Runtime → Change runtime type → T4 GPU.*

# Notebook 01 — Pretrained Segmentation with Cellpose-SAM (Lab 1)

**Lab time.** 90 minutes.
**Tool.** Cellpose-SAM (Cellpose v4 — Pachitariu, Rariden, Stringer, 2025).

**Learning goals.**

1. Run pretrained Cellpose on canonical microscopy data.
2. Inspect the segmentation output critically.
3. Identify cases where the model succeeds and where it fails.
4. Quantify simple morphological features from the segmentation.
5. Save outputs for the Lab 2 validation notebook.

We will deliberately try the model on both an easy case (where it shines) and a harder case (where it struggles). The failure analysis is half the lesson.

> **A note on the form widgets.** Several cells below use `#@param` comments. In **Google Colab** these render as interactive form widgets (sliders, dropdowns, checkboxes) at the top of the cell. In **other environments** (JupyterLab, VS Code, the JB rendered HTML) they appear as plain Python comments — edit the values directly and re-run the cell.

## Setup

GPU is recommended. On Colab: `Runtime → Change runtime type → GPU` (T4 free tier is fine). Cellpose runs on CPU too; just slower.

In [ ]:
import sys, os
IN_COLAB = "google.colab" in sys.modules

# Cellpose isn't in Colab's default environment. Installing it pulls a newer
# numpy as a transitive dep, which (a) conflicts with Colab's preinstalled
# numba 0.60 (which requires numpy<2.1), and (b) leaves Python's already-
# loaded scikit-image in a stale-ABI state ("cannot import name '_center'
# from 'numpy._core.umath'"). Three-part fix:
#   (1) cap numpy at <2.1 so it stays compatible with Colab's numba;
#   (2) reinstall scikit-image and tifffile alongside cellpose so they all
#       agree on the resulting numpy ABI;
#   (3) force-restart the kernel so the freshly-installed modules actually
#       load (without restart, Python keeps using the OLD cached numpy).
# Colab auto-reconnects; click 'Run all' once more after the restart and the
# cell falls through to the standard imports.
try:
    import cellpose  # noqa: F401
except ImportError:
    if not IN_COLAB:
        raise RuntimeError("cellpose not installed. Run: pip install 'cellpose>=3.0'")
    print("Installing cellpose. The kernel will restart; click 'Run all' again after reconnect.")
    get_ipython().run_line_magic(
        "pip",
        "install --quiet --upgrade 'cellpose>=3.0' 'numpy<2.1' scikit-image tifffile",
    )
    os.kill(os.getpid(), 9)   # SIGKILL — Colab auto-restarts the runtime

import numpy as np
import matplotlib.pyplot as plt
from skimage import io as skio
from skimage.measure import regionprops_table
import pandas as pd
print("Imports OK.")

In [ ]:
# Check GPU availability (informational; Cellpose handles fallback automatically)
try:
    import torch
    print("torch    :", torch.__version__)
    print("CUDA available :", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("Device         :", torch.cuda.get_device_name(0))
except ImportError:
    print("torch not installed (cellpose will install it)")

<!-- DATA-DECISION -->
## Choose your data source

This notebook can run on four kinds of data — pick one in the cell below.

- **MABC hosted** *(default)* — curated samples produced by the Mt Sinai Microscopy and Advanced Bioimaging Core, sized and formatted for this notebook. Fast, reproducible, license-clean.
- **Canonical** — fetch the published reference dataset (BBBC020 — Murine bone-marrow derived macrophages) from its upstream source. Slower but pedagogically the same.
- **Synthetic** — generate the toy data the notebook was originally written against. Always works, even offline. The "what you should be seeing" callouts further down were written for this path.
- **My own data → see T0** — opens the [data sources reference notebook](https://microscopy-core-ismms.github.io/ImageAnalysisCourse/notebooks/00_data_sources.html) with copy-pasteable blocks (local files, Google Drive, public URL, etc.).

If the chosen tier fails (network down, file missing), the loader falls through automatically: MABC → canonical → synthetic. Every cell prints which tier won.

- **Source:** [https://bbbc.broadinstitute.org/BBBC020](https://bbbc.broadinstitute.org/BBBC020)
- **License:** CC0
- **Citation:** Ljosa et al., Nature Methods, 2012 — BBBC020


In [ ]:
# DATA-DECISION
# @title Choose data source { run: "auto", display-mode: "form" }
DATA_SOURCE = "MABC hosted"  # @param ["MABC hosted", "Canonical (BBBC etc.)", "Synthetic", "My own data → see T0 notebook"]

import os, sys, traceback, tempfile, urllib.request, urllib.error, zipfile
import numpy as _np

NB_ID = '01_cellpose_segmentation'
MABC_URL = f"https://microscopy-core-ismms.github.io/ImageAnalysisCourse/data/mabc/{NB_ID}.npz"
CANONICAL_URL = 'https://data.broadinstitute.org/bbbc/BBBC020/BBBC020_v1_images.zip'
CANONICAL_NAME = 'BBBC020 — Murine bone-marrow derived macrophages'

real_imgs = None
real_filenames = None
real_metadata = None
real_labels = None
loaded_tier = None


def _try_mabc():
    """Fetch the MABC sample npz from gh-pages. Returns (imgs, filenames, metadata, labels).
    `labels` is None unless the npz includes a `labels.npy` (paired masks, target channels, etc.)."""
    cache = os.path.join(tempfile.gettempdir(), os.path.basename(MABC_URL))
    if not os.path.exists(cache):
        print(f"Fetching MABC sample: {MABC_URL}")
        urllib.request.urlretrieve(MABC_URL, cache)
    data = _np.load(cache, allow_pickle=True)
    imgs = list(data['images'])
    fnames = list(data['filenames']) if 'filenames' in data.files else [f"mabc_{i}" for i in range(len(imgs))]
    try:
        meta = data['metadata'].item() if 'metadata' in data.files else {}
    except Exception:
        meta = {}
    labels = list(data['labels']) if 'labels' in data.files else None
    return imgs, fnames, meta, labels


def _try_canonical():
    """Existing zip-based fetch from BBBC / GigaDB. Same logic as the prior architecture."""
    cache_zip = os.path.join(tempfile.gettempdir(), os.path.basename(CANONICAL_URL))
    cache_dir = cache_zip + "_extracted"
    if not os.path.exists(cache_zip):
        print(f"Fetching canonical: {CANONICAL_NAME} (this can take 10-60 s)...")
        urllib.request.urlretrieve(CANONICAL_URL, cache_zip)
        print(f"  cached at {cache_zip} ({os.path.getsize(cache_zip)/1e6:.1f} MB)")
    if not os.path.isdir(cache_dir):
        os.makedirs(cache_dir, exist_ok=True)
        with zipfile.ZipFile(cache_zip) as zf:
            zf.extractall(cache_dir)
    try:
        import tifffile
        _read = lambda p: tifffile.imread(p)
    except ImportError:
        from PIL import Image
        _read = lambda p: _np.array(Image.open(p))
    exts = ('.tif', '.tiff', '.TIF', '.TIFF', '.png', '.PNG')
    paths = []
    for root, _, files in os.walk(cache_dir):
        for fn in files:
            if fn.endswith(exts):
                paths.append(os.path.join(root, fn))
    paths.sort()
    paths = paths[:8]
    imgs = [_read(p) for p in paths]
    fnames = [os.path.relpath(p, cache_dir) for p in paths]
    meta = {"source": CANONICAL_NAME, "url": CANONICAL_URL, "tier": "canonical"}
    return imgs, fnames, meta


# Tier resolution
if DATA_SOURCE == "My own data → see T0 notebook":
    print("Open the T0 notebook for copy-paste data-loading blocks:")
    print(f"  https://microscopy-core-ismms.github.io/ImageAnalysisCourse/notebooks/00_data_sources.html")
    print("Once your images are loaded into a list called `real_imgs`, re-run the rest of this notebook.")

elif DATA_SOURCE == "Synthetic":
    print("Synthetic-only mode: skipping all real-data tiers; the synthetic generation cell below will run.")

else:
    if DATA_SOURCE == "MABC hosted":
        try:
            real_imgs, real_filenames, real_metadata, real_labels = _try_mabc()
            loaded_tier = "MABC"
        except (urllib.error.HTTPError, urllib.error.URLError, FileNotFoundError):
            print("MABC sample not yet available; falling through to canonical.")
        except Exception:
            print("MABC fetch raised an unexpected error; falling through to canonical.")
            traceback.print_exc(limit=2)

    if real_imgs is None:
        try:
            real_imgs, real_filenames, real_metadata = _try_canonical()
            real_labels = None
            loaded_tier = "Canonical"
        except Exception:
            print("Canonical fetch failed; the synthetic-generation cell below will run as the final fallback.")
            traceback.print_exc(limit=2)


# Bind working variables and display the loaded grid (only if a real tier won).
if real_imgs is not None:
    print(f"\nLoaded {len(real_imgs)} images from tier: {loaded_tier}.")
    if real_metadata:
        print(f"  source: {real_metadata.get('source', '(unknown)')}")
        print(f"  license: {real_metadata.get('license', 'see source')}")
        if 'citation' in real_metadata:
            print(f"  cite: {real_metadata['citation']}")

    # ---- Per-NB binding (lifted from the prior architecture's swap_code) ----
    if real_imgs and len(real_imgs) >= 2:
        img_easy = real_imgs[0]
        img_hard = real_imgs[1]
        img_easy_synth = real_imgs[0]
        img_hard_synth = real_imgs[1]
        img = real_imgs[0]
        print('img_easy / img_hard / *_synth now bound to BBBC020 real images (real_imgs[0] and real_imgs[1]).')
        print("NOTE: 'What you should be seeing' callouts were written for synthetic; counts and shapes will differ.")
    else:
        print('real_imgs is None or has <2 images; staying with synthetic.')


    # ---- Universal display grid ----
    try:
        import matplotlib.pyplot as _plt
        _n_show = min(8, len(real_imgs))
        _ncols = 4
        _nrows = (_n_show + _ncols - 1) // _ncols
        _fig, _axes = _plt.subplots(_nrows, _ncols, figsize=(3 * _ncols, 3 * _nrows))
        _ax_iter = list(_axes.flat) if hasattr(_axes, 'flat') else [_axes]
        for _i, _ax in enumerate(_ax_iter[:_n_show]):
            _disp = _np.asarray(real_imgs[_i]).astype(float)
            if _disp.ndim == 3:
                if _disp.shape[-1] in (3, 4):
                    pass  # RGB(A)
                else:
                    _disp = _disp.mean(axis=-1) if _disp.shape[-1] < min(_disp.shape[:2]) else _disp[_disp.shape[0]//2]
            _vmin, _vmax = _np.percentile(_disp, [1, 99])
            if _vmax <= _vmin:
                _vmin, _vmax = float(_disp.min()), float(_disp.max())
                if _vmax <= _vmin:
                    _vmax = _vmin + 1.0
            _cmap = None if (_disp.ndim == 3 and _disp.shape[-1] in (3, 4)) else 'gray'
            _ax.imshow(_disp, cmap=_cmap, vmin=_vmin, vmax=_vmax)
            _fn = (real_filenames[_i] if real_filenames and _i < len(real_filenames) else f'img {_i}')
            _ax.set_title(f"{loaded_tier}: {str(_fn)[:32]}", fontsize=8)
            _ax.axis('off')
        for _ax in _ax_iter[_n_show:]:
            _ax.axis('off')
        _plt.tight_layout(); _plt.show()
    except Exception:
        print("Could not render preview grid; data is still in real_imgs.")
        traceback.print_exc(limit=2)
else:
    print("Real-data tiers did not produce data. The synthetic-generation cell below will run.")


## Generate the working dataset

Two synthetic images we control fully: an **easy** case (round, well-separated cells, similar to Cellpose's training distribution) and a **harder** case (irregular shapes, denser packing — engineered to fall outside the easy distribution). Generating them inline keeps the notebook self-contained — no external URLs to rot.

Real canonical data comes in a later cell — once we've established the basic pattern, we apply Cellpose-SAM to three publicly available scikit-image research datasets.

In [ ]:
# GATE-SYNTHETIC: synthetic generation runs only as a fallback when real data is not loaded.
if globals().get('real_imgs') is not None:
    print('Synthetic generation skipped — real data is loaded into the working variables.')
else:
    from scipy.ndimage import gaussian_filter

    def make_easy_image(seed=0, size=200, n_cells=12):
        """Easy case: round, well-separated cells. Cellpose-friendly."""
        rng = np.random.default_rng(seed)
        img = np.zeros((size, size), dtype=float)
        centers = rng.uniform(20, size-20, (n_cells, 2))
        radii = rng.uniform(10, 18, n_cells)
        for (cy, cx), r in zip(centers, radii):
            Y, X = np.ogrid[:size, :size]
            img[(Y - cy)**2 + (X - cx)**2 <= r**2] = rng.uniform(0.6, 1.0)
        img = gaussian_filter(img, sigma=1.0) + rng.normal(0, 0.05, img.shape)
        return (np.clip(img, 0, 1) * 255).astype(np.uint8)

    def make_hard_image(seed=2, size=200, n_cells=18):
        """Harder case: irregular, dense, varying intensity. Out of distribution."""
        rng = np.random.default_rng(seed)
        img = np.zeros((size, size), dtype=float)
        centers = rng.uniform(15, size-15, (n_cells, 2))
        for cy, cx in centers:
            Y, X = np.ogrid[:size, :size]
            ry = rng.uniform(6, 14)
            rx = rng.uniform(6, 14) * rng.uniform(0.7, 1.4)
            ang = rng.uniform(0, np.pi)
            Yr = (Y - cy) * np.cos(ang) + (X - cx) * np.sin(ang)
            Xr = -(Y - cy) * np.sin(ang) + (X - cx) * np.cos(ang)
            img[(Yr / ry)**2 + (Xr / rx)**2 <= 1] = rng.uniform(0.3, 0.9)
        img = gaussian_filter(img, sigma=0.8) + rng.normal(0, 0.08, img.shape)
        return (np.clip(img, 0, 1) * 255).astype(np.uint8)

    # Save as PNG so the rest of the notebook can read them as files
    img_easy_synth = make_easy_image()
    img_hard_synth = make_hard_image()
    skio.imsave("sample_easy.png", img_easy_synth)
    skio.imsave("sample_hard.png", img_hard_synth)
    print(f"sample_easy.png: shape={img_easy_synth.shape} dtype={img_easy_synth.dtype}")
    print(f"sample_hard.png: shape={img_hard_synth.shape} dtype={img_hard_synth.dtype}")
    # Display the freshly generated synthetic train pair
    import matplotlib.pyplot as _plt
    _fig, _axes = _plt.subplots(1, 2, figsize=(10, 5))
    _axes[0].imshow(img_easy_synth, cmap='gray'); _axes[0].set_title('img_easy_synth (synthetic)', fontsize=10); _axes[0].axis('off')
    _axes[1].imshow(img_hard_synth, cmap='gray'); _axes[1].set_title('img_hard_synth (synthetic)', fontsize=10); _axes[1].axis('off')
    _plt.tight_layout(); _plt.show()


## Load Cellpose-SAM

We initialize the default Cellpose model. In Cellpose v4 (Cellpose-SAM) the default model uses the SAM transformer backbone for improved generalization.

In [ ]:
from cellpose import models, core

# core.use_gpu() picks the best available device automatically
use_gpu = core.use_gpu()
print(f"Using GPU: {use_gpu}")

# CellposeModel is the v3+/v4 entry point
model = models.CellposeModel(gpu=use_gpu)
print("Model loaded.")

## Run on the easy image

**What's happening here.** We feed the synthetic easy image into Cellpose-SAM with `diameter=None` (auto-estimate). The function returns three things: `masks` (an integer label image, one value per detected cell), `flows` (the internal gradient prediction Cellpose used to recover instances), and `styles` (a learned image embedding — useful for human-in-the-loop training; we ignore it today).

**Predict before running.** The synthetic easy image has 12 cells drawn into it. With auto-diameter and a clean image, what do you expect Cellpose-SAM to return?

- (a) Exactly 12 — the model recovers every cell.
- (b) 10–14 — close to 12 with a couple of merges or splits at the boundary.
- (c) Many more than 12 — the model over-segments noisy regions.
- (d) Far fewer than 12 — the model is under-confident on synthetic data.

Note your answer mentally. Run the next cells, then check.

In [ ]:
# SLIDER-RANGE-FIX: read PNG only if real data is not loaded (real_imgs is None).
if globals().get("real_imgs") is None:
    img_easy = skio.imread("sample_easy.png")
# If the image is RGB, Cellpose expects channels in a specific format
print("Easy image shape:", img_easy.shape, "dtype:", img_easy.dtype)

# Cellpose API: model.eval returns (masks, flows, styles)
# diameter=None lets the model auto-estimate cell size
masks_easy, flows_easy, styles_easy = model.eval(
    img_easy,
    diameter=None,
    channels=[0, 0],  # grayscale (or use [2, 1] for RGB cyto + nuclei)
)
print(f"Found {masks_easy.max()} objects in the easy image.")

## Visualize the easy result

In [ ]:
from matplotlib.colors import ListedColormap

# Build a colormap for instance labels (background black, then qualitative)
n_lbl = max(masks_easy.max(), 1)
cmap = ListedColormap(['black'] + plt.get_cmap('tab20')(np.linspace(0, 1, 20)).tolist())

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
axes[0].imshow(img_easy, cmap='gray'); axes[0].set_title("Image")
axes[1].imshow(masks_easy, cmap=cmap, vmin=0, vmax=20); axes[1].set_title(f"Masks ({n_lbl} objects)")
axes[2].imshow(img_easy, cmap='gray')
axes[2].imshow(np.where(masks_easy > 0, masks_easy, np.nan), cmap=cmap, alpha=0.5)
axes[2].set_title("Overlay")
for a in axes:
    a.axis("off")
plt.tight_layout(); plt.show()

**What you should be seeing.** Cellpose-SAM typically recovers 11–13 objects on this image (answer (b)). If you predicted (a) — exact recovery — you assumed the model is perfect; even on canonical-looking data there is small variance from boundary pixels. If you predicted (c) — many more — you over-weighted the noise we added; Cellpose's smoothing prior handles low-level pixel noise well. If (d), the model was probably more confident than you expected on synthetic data because the synthetic generator hit Cellpose's training distribution by accident.

**The habit to build.** Compare predictions to your own observations every time. In Lab 2 we'll put numbers on this comparison; for now, name out loud which cells the model missed, merged, or split.

## Now try the harder image

**What's happening here.** Same model, same call, different input. The harder image has 18 cells drawn with irregular shapes, dense packing, and varying intensity — engineered to fall outside what the model was trained on.

**Predict before running.** Look at `sample_hard.png` for a moment. The image has 18 cells. What do you expect Cellpose-SAM to return?

- (a) Roughly 18 — the model adapts to the harder morphology.
- (b) Considerably fewer — irregular shapes get merged into single masks.
- (c) Considerably more — the model over-segments the irregular shapes.
- (d) A mix — some merges and some over-segments balancing out.

The point is not to be right. The point is to commit to a hypothesis so you can update from the result.

In [ ]:
# SLIDER-RANGE-FIX: read PNG only if real data is not loaded (real_imgs is None).
if globals().get("real_imgs") is None:
    img_hard = skio.imread("sample_hard.png")
print("Hard image shape:", img_hard.shape)

masks_hard, flows_hard, styles_hard = model.eval(
    img_hard,
    diameter=None,
    channels=[0, 0],
)
print(f"Found {masks_hard.max()} objects in the harder image.")

In [ ]:
n_lbl = max(masks_hard.max(), 1)
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
axes[0].imshow(img_hard, cmap='gray'); axes[0].set_title("Image (harder)")
axes[1].imshow(masks_hard, cmap=cmap, vmin=0, vmax=20); axes[1].set_title(f"Masks ({n_lbl} objects)")
axes[2].imshow(img_hard, cmap='gray')
axes[2].imshow(np.where(masks_hard > 0, masks_hard, np.nan), cmap=cmap, alpha=0.5)
axes[2].set_title("Overlay")
for a in axes:
    a.axis("off")
plt.tight_layout(); plt.show()

**What you should be seeing.** On the harder image Cellpose-SAM typically returns somewhere between 8 and 14 objects — a *mix* of merges and splits (answer (d) is closest to truth). The merges happen where dense, irregular shapes touch; the splits happen where the model treats one elongated cell as two. This is the most insidious failure mode of pretrained models on OOD data: counts that *look* plausible because the failure types cancel out, hiding the actual segmentation errors.

**The lesson, before we quantify.** Pretrained models on OOD data produce confidently-wrong output. Counts alone won't tell you that — you need per-object metrics (Lab 2). Note this gap; it's the central pedagogy of the workshop.

## Quantify per-object features

In [ ]:
def feature_table(masks, name):
    if masks.max() == 0:
        return pd.DataFrame()
    props = regionprops_table(
        masks,
        properties=['label', 'area', 'equivalent_diameter', 'eccentricity', 'centroid'],
    )
    df = pd.DataFrame(props)
    df['image'] = name
    return df

df_easy = feature_table(masks_easy, "easy")
df_hard = feature_table(masks_hard, "hard")
df_all = pd.concat([df_easy, df_hard], ignore_index=True)

print("Per-image summary:")
print(df_all.groupby('image').agg(
    n_objects=('label', 'count'),
    mean_area=('area', 'mean'),
    mean_diam=('equivalent_diameter', 'mean'),
).round(2))

## Compare feature distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for img_name, color in [("easy", "#4C72B0"), ("hard", "#C44E52")]:
    sub = df_all[df_all['image'] == img_name]
    axes[0].hist(sub['area'], bins=20, alpha=0.6, label=img_name, color=color)
    axes[1].hist(sub['equivalent_diameter'], bins=20, alpha=0.6, label=img_name, color=color)
axes[0].set_xlabel("Area (pixels)"); axes[0].set_ylabel("Count"); axes[0].legend(); axes[0].set_title("Cell area distribution")
axes[1].set_xlabel("Equivalent diameter (pixels)"); axes[1].set_ylabel("Count"); axes[1].legend(); axes[1].set_title("Cell diameter distribution")
plt.tight_layout(); plt.show()

**Reflection.** Are the distributions plausible for what you saw in the images? If the harder image's distribution looks *very* different from the easy one's, ask: is that biology, or is that segmentation failure?

This is the question Lab 2 will let you answer quantitatively against ground truth.

## Real canonical microscopy data

Now that the basic pattern works on synthetic images, let's apply Cellpose-SAM to **real** research data. We use three datasets bundled with scikit-image — they are real fluorescence-style microscopy images, included in the package, so no external download is needed.

In [ ]:
from skimage import data

img_cell    = data.cell()                   # single fluorescence cell (2D grayscale)
img_mitosis = data.human_mitosis()          # real mitosis fluorescence (2D grayscale)
img_3d      = data.cells3d()                # 3D fluorescence stack: (z, channel, y, x)
img_3d_slice = img_3d[30, 1]                # mid-Z slice, nuclei channel

real_data = {
    "data.cell()":            img_cell,
    "data.human_mitosis()":   img_mitosis,
    "data.cells3d() (slice)": img_3d_slice,
}

for name, img in real_data.items():
    print(f"{name:<26}: shape={img.shape}  dtype={img.dtype}  range=[{img.min()}, {img.max()}]")

**Try this — predict before you reveal.** Looking at the dimensions and value ranges above, which image do you expect Cellpose-SAM to handle best? Worst? Why?

In [ ]:
# Display the three real images
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, img) in zip(axes, real_data.items()):
    p1, p99 = np.percentile(img, [1, 99])
    ax.imshow(img, cmap='gray', vmin=p1, vmax=p99)
    ax.set_title(name); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Run Cellpose-SAM on each, with auto-diameter
from matplotlib.colors import ListedColormap
cmap = ListedColormap(['black'] + plt.get_cmap('tab20')(np.linspace(0, 1, 20)).tolist())

real_masks = {}
for name, img in real_data.items():
    masks, _, _ = model.eval(img, diameter=None, channels=[0, 0])
    real_masks[name] = masks
    print(f"{name:<26}: {masks.max()} objects detected")

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for col, (name, img) in enumerate(real_data.items()):
    masks = real_masks[name]
    p1, p99 = np.percentile(img, [1, 99])
    axes[0, col].imshow(img, cmap='gray', vmin=p1, vmax=p99)
    axes[0, col].set_title(name); axes[0, col].axis('off')
    axes[1, col].imshow(img, cmap='gray', vmin=p1, vmax=p99)
    axes[1, col].imshow(np.where(masks > 0, masks, np.nan), cmap=cmap, alpha=0.5, vmin=0, vmax=20)
    axes[1, col].set_title(f"{masks.max()} objects"); axes[1, col].axis('off')
plt.tight_layout(); plt.show()

**Discussion.** Compare your predictions to what you saw. Where did Cellpose-SAM perform well? Where did it struggle? The auto-diameter estimator is reasonable but not perfect — for any specific image, you may know the true diameter and providing it can help.

This brings us to the customization workshop.

## Customization workshop

Cellpose-SAM has knobs. Most users leave them on default and live with the result. The next four cells walk through the four most consequential parameters and show how output changes when you turn each knob. The lesson is mechanical: build intuition for *which knob does what*. The harder lesson — *parameters don't fix fundamentally OOD data* — is at the end.

For each knob: predict before you run, observe how the count changes, ask whether the change made the segmentation more or less *biologically* correct.

### Knob 1 — `diameter`

The most important parameter. Cellpose needs to know cell size in pixels. Default `diameter=None` lets the model auto-estimate. If you guess wrong by a factor of 2× either way, you get nonsense.

**Try this** — predict, then run:
- Auto (`None`) — should land near the true diameter (~22 px for our easy synthetic case).
- Very small (`5 px`) — predict: more objects, fewer, or different shape?
- Very large (`50 px`) — predict the same.

In [ ]:
img = img_easy_synth  # synthetic easy case (12 cells, true diameter ~22 px)

diameter_settings = [("auto (None)", None), ("small (5 px)", 5), ("large (50 px)", 50)]
diameter_results = {}
for label, dia in diameter_settings:
    masks, _, _ = model.eval(img, diameter=dia, channels=[0, 0])
    diameter_results[label] = (masks, masks.max())
    print(f"diameter={label:<14}: {masks.max()} objects")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, (label, (masks, n)) in zip(axes, diameter_results.items()):
    ax.imshow(img, cmap='gray')
    ax.imshow(np.where(masks > 0, masks, np.nan), cmap=cmap, alpha=0.5, vmin=0, vmax=20)
    ax.set_title(f"diameter = {label}\n{n} objects"); ax.axis('off')
plt.tight_layout(); plt.show()

**Lesson.** Diameter is the most important knob. Wrong diameter → wrong segmentation, even on a clean image. Auto-estimation is reasonable; explicit values help when you know the truth. Guessing badly produces confidently-wrong output.

**Free exploration — slide and re-run.** The cell below exposes `diameter` as an interactive slider in Colab. Pick a value, run the cell, observe. Try crossing the *true* value (≈22 px for the easy synthetic case) from below and from above; note where the count plateaus and where it explodes.

In [ ]:
# @title Diameter exploration { run: "auto" }
diameter = 22  # @param {type: "slider", min: 5, max: 60, step: 1}

img = img_easy_synth
masks, _, _ = model.eval(img, diameter=diameter, channels=[0, 0])
print(f"diameter={diameter}: {masks.max()} objects")

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.imshow(img, cmap='gray')
ax.imshow(np.where(masks > 0, masks, np.nan), cmap=cmap, alpha=0.5, vmin=0, vmax=20)
ax.set_title(f"diameter = {diameter}  →  {masks.max()} objects"); ax.axis('off')
plt.tight_layout(); plt.show()

### Knob 2 — model variant

Cellpose ships several specialized models. The default `cyto3` is for cytoplasm-stained cells. The `nuclei` model is trained on nuclei specifically. Same image, different model, different inductive bias.

**Try this** — apply both `cyto3` and `nuclei` to `img_mitosis` (which IS nuclei). Predict which model performs better before running.

In [ ]:
from cellpose import models as cp_models

# Load both model variants
model_cyto3   = cp_models.CellposeModel(gpu=use_gpu, model_type="cyto3")
model_nuclei  = cp_models.CellposeModel(gpu=use_gpu, model_type="nuclei")

img_for_test = img_mitosis  # nuclei image; should favor 'nuclei' model

masks_cyto3,  _, _ = model_cyto3.eval(img_for_test,  diameter=None, channels=[0, 0])
masks_nuclei, _, _ = model_nuclei.eval(img_for_test, diameter=None, channels=[0, 0])

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
p1, p99 = np.percentile(img_for_test, [1, 99])
axes[0].imshow(img_for_test, cmap='gray', vmin=p1, vmax=p99); axes[0].set_title("Image (mitosis)")
axes[1].imshow(img_for_test, cmap='gray', vmin=p1, vmax=p99)
axes[1].imshow(np.where(masks_cyto3 > 0, masks_cyto3, np.nan), cmap=cmap, alpha=0.5, vmin=0, vmax=20)
axes[1].set_title(f"cyto3 model: {masks_cyto3.max()} objects")
axes[2].imshow(img_for_test, cmap='gray', vmin=p1, vmax=p99)
axes[2].imshow(np.where(masks_nuclei > 0, masks_nuclei, np.nan), cmap=cmap, alpha=0.5, vmin=0, vmax=20)
axes[2].set_title(f"nuclei model: {masks_nuclei.max()} objects")
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

**Lesson.** Model choice matters. Use the model trained on data closest to yours — `nuclei` for nuclei, `cyto`/`cyto3` for cytoplasm. The model-type registry is a real research signal: if no specialized model exists for your sample type, your task is harder.

**Free exploration — pick a model, pick an image.** Two dropdowns: which Cellpose-SAM variant, and which test image. The lesson lands hardest when the two disagree (e.g., `cyto3` on a nuclei image, or `nuclei` on a cytoplasm image).

In [ ]:
# @title Model-variant exploration { run: "auto" }
model_variant = "cyto3"  # @param ["cyto3", "nuclei"]
test_image = "human_mitosis (nuclei)"  # @param ["human_mitosis (nuclei)", "cell (cytoplasm)", "cells3d slice (nuclei)"]

_test_lookup = {
    "human_mitosis (nuclei)":   img_mitosis,
    "cell (cytoplasm)":         img_cell,
    "cells3d slice (nuclei)":   img_3d_slice,
}
img_for_test = _test_lookup[test_image]
m_variant = cp_models.CellposeModel(gpu=use_gpu, model_type=model_variant)
masks_v, _, _ = m_variant.eval(img_for_test, diameter=None, channels=[0, 0])

p1, p99 = np.percentile(img_for_test, [1, 99])
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].imshow(img_for_test, cmap='gray', vmin=p1, vmax=p99); axes[0].set_title(test_image); axes[0].axis('off')
axes[1].imshow(img_for_test, cmap='gray', vmin=p1, vmax=p99)
axes[1].imshow(np.where(masks_v > 0, masks_v, np.nan), cmap=cmap, alpha=0.5, vmin=0, vmax=20)
axes[1].set_title(f"{model_variant}: {masks_v.max()} objects"); axes[1].axis('off')
plt.tight_layout(); plt.show()

### Knob 3 — `flow_threshold`

Cellpose internally predicts gradient flows; cells are recovered from the flow. `flow_threshold` gates how strict the quality check on each candidate cell is. Default `0.4`. Lower = stricter (rejects more borderline cells). Higher = lenient (accepts more, including likely mistakes).

**Try this** — apply three values to the easy case. Predict the count direction.

In [ ]:
img = img_easy_synth
flow_results = {}
for ft in [0.0, 0.4, 0.9]:
    masks, _, _ = model.eval(img, diameter=None, channels=[0, 0], flow_threshold=ft)
    flow_results[ft] = masks
    print(f"flow_threshold={ft}: {masks.max()} objects")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, (ft, masks) in zip(axes, flow_results.items()):
    ax.imshow(img, cmap='gray')
    ax.imshow(np.where(masks > 0, masks, np.nan), cmap=cmap, alpha=0.5, vmin=0, vmax=20)
    ax.set_title(f"flow_threshold = {ft}\n{masks.max()} objects"); ax.axis('off')
plt.tight_layout(); plt.show()

**Lesson.** `flow_threshold` trades off completeness vs. purity. Strict (low value) drops borderline cells; lenient (high value) catches more cells but also more spurious detections.

**Free exploration — slide the strictness.** Find the value where the count starts to *visibly* over-count (spurious detections appear in the noise) and the value where it starts to drop *real* cells. Those two values bracket the usable range for this image. The default `0.4` is a reasonable middle.

In [ ]:
# @title flow_threshold exploration { run: "auto" }
flow_threshold = 0.4  # @param {type: "slider", min: 0.0, max: 1.0, step: 0.05}

img = img_easy_synth
masks, _, _ = model.eval(img, diameter=None, channels=[0, 0], flow_threshold=flow_threshold)

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.imshow(img, cmap='gray')
ax.imshow(np.where(masks > 0, masks, np.nan), cmap=cmap, alpha=0.5, vmin=0, vmax=20)
ax.set_title(f"flow_threshold = {flow_threshold}  →  {masks.max()} objects"); ax.axis('off')
plt.tight_layout(); plt.show()

### Knob 4 — `cellprob_threshold`

Per-pixel cell-probability threshold. Default `0`. Higher = more confident (smaller masks, fewer cells). Lower = less confident (larger masks, more cells, potentially merging neighbors).

**Try this** — apply three values to the easy case. Watch how mask *boundaries* change, not just object counts.

In [ ]:
img = img_easy_synth
cprob_results = {}
for cpt in [-2, 0, 2]:
    masks, _, _ = model.eval(img, diameter=None, channels=[0, 0], cellprob_threshold=cpt)
    cprob_results[cpt] = masks
    print(f"cellprob_threshold={cpt}: {masks.max()} objects")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, (cpt, masks) in zip(axes, cprob_results.items()):
    ax.imshow(img, cmap='gray')
    ax.imshow(np.where(masks > 0, masks, np.nan), cmap=cmap, alpha=0.5, vmin=0, vmax=20)
    ax.set_title(f"cellprob_threshold = {cpt}\n{masks.max()} objects"); ax.axis('off')
plt.tight_layout(); plt.show()

**Lesson.** `cellprob_threshold` controls how confident the model has to be that a pixel is foreground. Lower thresholds expand masks (sometimes merging adjacent cells); higher thresholds shrink them (sometimes losing parts of cells).

**Free exploration — slide the per-pixel confidence.** Watch *boundaries*, not just counts. Where do masks expand into the background? Where do adjacent cells merge into a single mask? Those are the two failure modes this knob trades between.

In [ ]:
# @title cellprob_threshold exploration { run: "auto" }
cellprob_threshold = 0.0  # @param {type: "slider", min: -6.0, max: 6.0, step: 0.5}

img = img_easy_synth
masks, _, _ = model.eval(img, diameter=None, channels=[0, 0], cellprob_threshold=cellprob_threshold)

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.imshow(img, cmap='gray')
ax.imshow(np.where(masks > 0, masks, np.nan), cmap=cmap, alpha=0.5, vmin=0, vmax=20)
ax.set_title(f"cellprob_threshold = {cellprob_threshold}  →  {masks.max()} objects"); ax.axis('off')
plt.tight_layout(); plt.show()

## Failure-mode finale — can we rescue the hard case with parameters?

**What's happening here.** The hard synthetic image (`img_hard_synth`) is fundamentally out-of-distribution: irregular shapes, dense packing, anisotropic morphology. It does not look like Cellpose's training distribution. Now that you've seen each knob individually, the next cell tries three plausible parameter combinations on the hard case and shows the results side-by-side.

**Predict before running.** The hard image has 18 cells. Across the three configs the next cell tries — *auto everything*, *small + lenient*, *moderate + strict* — what do you expect?

- (a) At least one config recovers something close to 18 with biologically plausible boundaries — the failure was a tuning problem.
- (b) All three configs return wildly different counts, none close to 18 — the failure is fundamental, not a tuning problem.
- (c) Counts converge near 18 but boundaries are wrong everywhere — counts can lie.
- (d) The "moderate + strict" config wins because strictness compensates for OOD-ness.

If you predicted (a), you're in a hopeful camp; if (b) or (c), you've internalized the lecture's "confidently wrong" framing. The result will tell you which.

In [ ]:
img = img_hard_synth

# Three plausible parameter combinations to try
configs = [
    {"label": "auto everything",          "diameter": None, "flow_threshold": 0.4, "cellprob_threshold": 0},
    {"label": "small + lenient",           "diameter": 8,    "flow_threshold": 0.7, "cellprob_threshold": -1},
    {"label": "moderate + strict",         "diameter": 14,   "flow_threshold": 0.2, "cellprob_threshold": 1},
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, cfg in zip(axes, configs):
    masks, _, _ = model.eval(
        img, channels=[0, 0],
        diameter=cfg["diameter"],
        flow_threshold=cfg["flow_threshold"],
        cellprob_threshold=cfg["cellprob_threshold"],
    )
    ax.imshow(img, cmap='gray')
    ax.imshow(np.where(masks > 0, masks, np.nan), cmap=cmap, alpha=0.5, vmin=0, vmax=20)
    ax.set_title(f"{cfg['label']}\n{masks.max()} objects"); ax.axis('off')
plt.tight_layout(); plt.show()

**What you should be seeing.** Three different parameter combinations produced three different counts on the hard image — and *none* of them recovers 18 cells with biologically plausible boundaries. Answer (b) or (c) was correct; answer (a) was wishful. **Different output ≠ correct output.** When the model has never seen anything like this sample type, no combination of knob settings will produce correct segmentation.

**The lesson.** Parameter tuning shifts output but cannot rescue fundamentally OOD data. The right move when you hit an OOD case is *not* harder parameter tuning. It is one of:

- *Use a different model* (cyto3, nuclei, livecell, custom) trained closer to your sample type.
- *Fine-tune* a model on a small set of your own labeled images (Cellpose 2.0+).
- *Use a foundation model with prompts* (segment-anything, μSAM) where the user provides per-image guidance — Lab 3b.
- *Accept the limit* and use a different segmentation paradigm entirely (classical, semi-automatic, etc.).

**Free exploration — try to break my claim.** The cell below exposes all four knobs at once on the hard image. Try to find a parameter combination that produces a *biologically* correct segmentation (not just a plausible count). If you find one, the claim above is wrong. If you can't, you've validated the lesson the hard way.

In [ ]:
# @title Failure-mode rescue attempt { run: "auto" }
diameter_hard = 12  # @param {type: "slider", min: 5, max: 60, step: 1}
flow_threshold_hard = 0.4  # @param {type: "slider", min: 0.0, max: 1.0, step: 0.05}
cellprob_threshold_hard = 0.0  # @param {type: "slider", min: -6.0, max: 6.0, step: 0.5}
model_variant_hard = "cyto3"  # @param ["cyto3", "nuclei"]

m = cp_models.CellposeModel(gpu=use_gpu, model_type=model_variant_hard)
masks, _, _ = m.eval(
    img_hard_synth, channels=[0, 0],
    diameter=diameter_hard,
    flow_threshold=flow_threshold_hard,
    cellprob_threshold=cellprob_threshold_hard,
)
print(f"d={diameter_hard}, ft={flow_threshold_hard}, cpt={cellprob_threshold_hard}, "
      f"model={model_variant_hard} → {masks.max()} objects (true: 18)")

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.imshow(img_hard_synth, cmap='gray')
ax.imshow(np.where(masks > 0, masks, np.nan), cmap=cmap, alpha=0.5, vmin=0, vmax=20)
ax.set_title(f"{masks.max()} objects (truth: 18)"); ax.axis('off')
plt.tight_layout(); plt.show()

Lab 2 will quantify what "correct" actually means against ground truth — turning this qualitative judgment into validation metrics.

## Save outputs for Lab 2

Lab 2 picks up these files. Don't skip this step.

In [ ]:
np.save("masks_easy.npy", masks_easy)
np.save("masks_hard.npy", masks_hard)
df_all.to_csv("features.csv", index=False)
print("Saved: masks_easy.npy, masks_hard.npy, features.csv")

## Going broader — community alternatives to Cellpose

Cellpose is one tool among many. Notebook 04 catalogs the full ZeroCostDL4Mic / DL4MicEverywhere / BioImage Model Zoo ecosystem. Methods worth comparing against Cellpose:

- **StarDist** — instance segmentation with star-convex shape priors. Often better for densely packed nuclei. *Notebook 04, catalog.*
- **Mesmer / DeepCell** — whole-cell (membrane + nucleus) segmentation; strong on tissue. *Notebook 04, catalog.*
- **U-Net (semantic)** — when you don't need instance separation. Lighter, faster. *Notebook 04, catalog.*
- **Browse the BioImage Model Zoo live** — the API in Notebook 04 lets you search the registry by task and load any model. If a pretrained segmentation model exists for your specific sample type, you save weeks.

Picking the right tool is half the work; the catalog makes that explicit.

## Closing reflection

Lab 1 demonstrated the full pretrained-model workflow: load, run, visualize, quantify. The harder case probably surfaced confident-but-wrong predictions — exactly the failure mode the morning lecture flagged.

Lab 2 will give you the validation tools to put numbers on those failures. See you there.

**Where to go next on your own:**
- Try Cellpose-SAM on one of your own images. Swap the file path in the load cell.
- Try `model.eval(img, channels=[2, 1])` on RGB images for cytoplasm + nuclei.
- Visit the [Cellpose GitHub repository](https://github.com/MouseLand/cellpose) for the model zoo and human-in-the-loop training.